# Liquidation Signal — Modular Feature Pipeline
## End-to-End: Dataset Construction → Model Training → Evaluation

This notebook runs the full pipeline:

```
raw market data
  → FeatureBlocks (declarative, composable)
  → TransformLayer (log / zscore / clamp)
  → Sampler (volume-clock / time-clock / trade-clock)
  → MarkoutLabeler (n-second PnL of executed trade)
  → DatasetOutput
  → ModelTraining
  → Results
```

**Key design principle:** features are _listed_, transforms are _declared as reusable operators_,
dataset construction is _deterministic and reproducible_, model training is _decoupled_.


In [ ]:
import logging
import warnings
warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)-8s %(message)s", datefmt="%H:%M:%S")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# ── Core pipeline abstractions ──────────────────────────────────────────────
from feature_pipeline import (
    # Feature blocks
    TopOfBookVolumeFeature, PriceZScoreFeature, RollingVolatilityFeature,
    BidAskSpreadFeature, TradePressureFeature, LiquidationPressureFeature,
    TimeFeature,
    # Transform layer
    Transform, TransformLayer, DEFAULT_TRANSFORMS,
    # Samplers
    VolumeClock, TimeClock, TradeClock,
    # Labeler
    MarkoutLabeler,
    # Orchestrators
    PipelineConfig, DatasetBuilder,
    PipelineState,
    # Model interface
    ModelSpec, train_model, evaluate_model,
)
from pipeline_runner import (
    load_data, make_pipeline_config, run_full_pipeline, print_report
)

print("✓ All imports OK")


## 1. Configuration

**Declarative pipeline spec** — change these variables, not the code below.


In [ ]:
# ── User-facing knobs ──────────────────────────────────────────────────────
DATA_DIR        = "data"          # path to parquet files
SYMBOLS         = ["btcusdt", "ethusdt"]
TAUS_S          = [30.0, 120.0, 300.0]  # markout horizons (seconds)
LIMIT           = 100_000         # trades per symbol per split (0 = all)
USE_ML          = True            # True = LightGBM, False = Ridge regression

# Sampler choice — swap string to change sampling strategy
SAMPLER_TYPE    = "volume"        # "volume" | "time" | "trade"
VOL_THRESHOLD   = 1_000_000.0    # USD notional per volume bar
TIME_INTERVAL_S = 60.0           # seconds per time bar
TRADE_NTH       = 1              # every Nth trade

# ── Declarative feature list (reorder, add, remove freely) ────────────────
FEATURE_BLOCKS = [
    TopOfBookVolumeFeature(),
    PriceZScoreFeature(windows_s=(10.0, 60.0, 300.0), clip=5.0),
    RollingVolatilityFeature(windows_s=(30.0, 300.0)),
    BidAskSpreadFeature(),
    TradePressureFeature(windows_s=(5.0, 30.0, 120.0)),
    LiquidationPressureFeature(windows_s=(5.0, 30.0)),
    TimeFeature(),
]

# ── Declarative transform list ─────────────────────────────────────────────
TRANSFORMS = TransformLayer([
    Transform(op="log",    columns=["signed_flow_5.0s", "signed_flow_30.0s", "signed_flow_120.0s"]),
    Transform(op="zscore", columns=[f"price_zscore_{w}s" for w in (10.0, 60.0, 300.0)]),
    Transform(op="clamp",  columns=[f"realized_vol_{w}s" for w in (30.0, 300.0)], clip_range=(-3.0, 3.0)),
    Transform(op="clamp",  columns=[f"taker_imbalance_{w}s" for w in (5.0, 30.0, 120.0)], clip_range=(-3.0, 3.0)),
])

print(f"Features: {[b.__class__.__name__ for b in FEATURE_BLOCKS]}")
print(f"Taus    : {TAUS_S} seconds")
print(f"Sampler : {SAMPLER_TYPE}")
print(f"Model   : {'LightGBM' if USE_ML else 'Ridge'}")


## 2. Build Pipeline Config & Dataset Builder

The `PipelineConfig` is the single declarative spec that drives the entire pipeline.


In [ ]:
config = PipelineConfig(
    features=FEATURE_BLOCKS,
    transforms=TRANSFORMS,
    sampler=VolumeClock(threshold=VOL_THRESHOLD) if SAMPLER_TYPE == "volume" else
            TimeClock(interval_s=TIME_INTERVAL_S) if SAMPLER_TYPE == "time" else
            TradeClock(nth=TRADE_NTH),
    labeler=MarkoutLabeler(tau_s=TAUS_S),
    apply_transforms=True,
)

builder = DatasetBuilder(config)
feature_names = builder.feature_names()
target_names  = builder.target_names()

print(f"Total features : {len(feature_names)}")
print(f"Feature columns: {feature_names}")
print(f"Target columns : {target_names}")


## 3. Load Data & Build Datasets


In [ ]:
train_datasets = {}
val_datasets   = {}

for symbol in SYMBOLS:
    print(f"\n{'─'*50}  {symbol.upper()}")

    # Train
    trades_tr, bbo_tr, liq_b_tr, liq_bb_tr = load_data(
        DATA_DIR, symbol, split="train", limit=LIMIT
    )
    print(f"  Train raw trades: {len(trades_tr):,}")

    if len(trades_tr) > 0:
        ds_tr = builder.build(trades_tr, bbo_tr, liq_b_tr, liq_bb_tr, symbol=symbol)
        train_datasets[symbol] = ds_tr
        print(f"  Train dataset   : {len(ds_tr):,} rows × {len(feature_names)} features")

    # Validation
    trades_val, bbo_val, liq_b_val, liq_bb_val = load_data(
        DATA_DIR, symbol, split="validation", limit=LIMIT
    )
    print(f"  Val raw trades  : {len(trades_val):,}")

    if len(trades_val) > 0:
        ds_val = builder.build(trades_val, bbo_val, liq_b_val, liq_bb_val, symbol=symbol)
        val_datasets[symbol] = ds_val
        print(f"  Val dataset     : {len(ds_val):,} rows × {len(feature_names)} features")

print("\n✓ Datasets built")


## 4. Dataset Inspection


In [ ]:
if train_datasets:
    sym0 = list(train_datasets.keys())[0]
    ds_sample = train_datasets[sym0]

    print(f"Shape: {ds_sample.shape}")
    print(f"\nTarget stats (train, {sym0}):")
    display(ds_sample[target_names].describe().round(4))

    print(f"\nFeature stats (first 8):")
    display(ds_sample[feature_names[:8]].describe().round(4))


### 4a. Target (PnL markout) Distribution


In [ ]:
if train_datasets:
    sym0 = list(train_datasets.keys())[0]
    ds_sample = train_datasets[sym0]

    fig, axes = plt.subplots(1, len(TAUS_S), figsize=(5*len(TAUS_S), 4))
    if len(TAUS_S) == 1:
        axes = [axes]

    for ax, tau in zip(axes, TAUS_S):
        col = f"pnl_bps_{int(tau)}s"
        if col not in ds_sample.columns:
            continue
        data = ds_sample[col].dropna()
        ax.hist(data.clip(-10, 10), bins=60, color="#2a78d6", alpha=0.8, edgecolor="none")
        ax.axvline(0, color="#e24b4a", linewidth=1.5, linestyle="--")
        ax.axvline(data.mean(), color="#1baf7a", linewidth=1.5, linestyle="--", label=f"mean={data.mean():.2f}")
        ax.set_title(f"PnL markout τ={int(tau)}s [{sym0.upper()}]", fontsize=12)
        ax.set_xlabel("bps")
        ax.set_ylabel("count")
        ax.legend(fontsize=9)

    plt.suptitle("Maker trade markout distribution (clipped ±10 bps)", y=1.02)
    plt.tight_layout()
    plt.savefig("target_distribution.png", dpi=120, bbox_inches="tight")
    plt.show()
    print("Saved: target_distribution.png")


### 4b. Feature–Target Correlation (IC)


In [ ]:
from scipy import stats

if train_datasets:
    sym0 = list(train_datasets.keys())[0]
    ds = train_datasets[sym0]
    primary_target = f"pnl_bps_{int(TAUS_S[0])}s"

    ics = {}
    for feat in feature_names:
        if feat not in ds.columns:
            continue
        valid = ds[[feat, primary_target]].dropna()
        if len(valid) < 10:
            continue
        ic, pval = stats.spearmanr(valid[feat], valid[primary_target])
        ics[feat] = ic

    ic_s = pd.Series(ics).sort_values(key=abs, ascending=False)

    fig, ax = plt.subplots(figsize=(10, max(4, len(ic_s)*0.35)))
    colors = ["#e34948" if v < 0 else "#1baf7a" for v in ic_s.values]
    ax.barh(ic_s.index[::-1], ic_s.values[::-1], color=colors[::-1], alpha=0.85)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_xlabel(f"IC (Spearman) vs {primary_target}")
    ax.set_title(f"Feature Information Coefficient [{sym0.upper()}]")
    plt.tight_layout()
    plt.savefig("feature_ic.png", dpi=120, bbox_inches="tight")
    plt.show()
    print("Saved: feature_ic.png")
    print("\nTop 5 features by |IC|:")
    print(ic_s.head(5).to_string())


## 5. Sampler Comparison

Demonstrate how the 3 sampling strategies produce different datasets from the same raw data.


In [ ]:
sym0 = SYMBOLS[0]
if sym0 in train_datasets:
    trades_tr, bbo_tr, liq_b_tr, liq_bb_tr = load_data(DATA_DIR, sym0, split="train", limit=LIMIT)

    results_by_sampler = {}
    for stype, sampler in [
        ("volume", VolumeClock(threshold=VOL_THRESHOLD)),
        ("time",   TimeClock(interval_s=TIME_INTERVAL_S)),
        ("trade",  TradeClock(nth=100)),          # every 100th trade
    ]:
        cfg = PipelineConfig(
            features=FEATURE_BLOCKS,
            transforms=TRANSFORMS,
            sampler=sampler,
            labeler=MarkoutLabeler(tau_s=[30.0]),
            apply_transforms=True,
        )
        b = DatasetBuilder(cfg)
        ds = b.build(trades_tr, bbo_tr, liq_b_tr, liq_bb_tr, symbol=sym0)
        results_by_sampler[stype] = ds
        print(f"  {stype:8s}: {len(ds):,} rows")

    print("\n✓ All samplers ran successfully")


## 6. Model Training

One model per **(symbol, τ)**. Training is decoupled from feature computation via `ModelSpec`.


In [ ]:
models  = {}
reports = {}

for symbol in SYMBOLS:
    if symbol not in train_datasets:
        continue

    ds_tr  = train_datasets[symbol]
    ds_val = val_datasets.get(symbol, pd.DataFrame())

    for tau in TAUS_S:
        target_col = f"pnl_bps_{int(tau)}s"
        if target_col not in ds_tr.columns:
            continue

        spec = ModelSpec(
            model_type="lgbm" if USE_ML else "linear",
            target_col=target_col,
            use_sample_weights=True,
        )

        model = train_model(ds_tr, feature_names, spec)
        models[(symbol, tau)] = model

        tr_m  = evaluate_model(model, ds_tr,  feature_names, spec)
        val_m = evaluate_model(model, ds_val, feature_names, spec) if len(ds_val) > 0 else {}

        reports[(symbol, tau)] = {"train": tr_m, "val": val_m}
        print(f"  [{symbol} τ={int(tau)}s] "
              f"train IC={tr_m.get('ic_spearman',0):.4f} wcorr={tr_m.get('weighted_corr',0):.4f} | "
              f"val IC={val_m.get('ic_spearman',0):.4f} wcorr={val_m.get('weighted_corr',0):.4f}")

print("\n✓ All models trained")


## 7. Results Summary


In [ ]:
rows = []
for (symbol, tau), r in sorted(reports.items()):
    tr = r.get("train", {})
    vl = r.get("val", {})
    rows.append({
        "Symbol":   symbol.upper(),
        "τ (s)":    int(tau),
        "Train IC": round(tr.get("ic_spearman", float("nan")), 4),
        "Train Corr": round(tr.get("weighted_corr", float("nan")), 4),
        "Val IC":   round(vl.get("ic_spearman", float("nan")), 4),
        "Val Corr": round(vl.get("weighted_corr", float("nan")), 4),
        "N train":  tr.get("n_samples", 0),
    })

df_results = pd.DataFrame(rows)
print("\n=== Training Results ===")
display(df_results)


## 8. Feature Importance (LightGBM)


In [ ]:
sym0 = SYMBOLS[0]
tau0 = TAUS_S[0]

if (sym0, tau0) in models and USE_ML:
    model = models[(sym0, tau0)]
    try:
        feat_imp = pd.Series(
            model.feature_importances_,
            index=[f for f in feature_names if f in train_datasets[sym0].columns]
        ).sort_values(ascending=False)

        fig, ax = plt.subplots(figsize=(10, max(4, len(feat_imp)*0.35)))
        feat_imp[::-1].plot.barh(ax=ax, color="#2a78d6", alpha=0.85)
        ax.set_title(f"Feature Importance [{sym0.upper()}, τ={int(tau0)}s]")
        ax.set_xlabel("Importance")
        plt.tight_layout()
        plt.savefig("feature_importance.png", dpi=120, bbox_inches="tight")
        plt.show()
        print("Saved: feature_importance.png")
        print("\nTop 10:")
        print(feat_imp.head(10).to_string())
    except AttributeError:
        print("Feature importances not available for this model type")
else:
    print("LightGBM model not available (set USE_ML=True and run training cell)")


## 9. Prediction Quality — Quantile PnL Chart

Sort validation samples by predicted score → check if top quantiles have better realized PnL.


In [ ]:
sym0 = SYMBOLS[0]
tau0 = TAUS_S[0]
target_col = f"pnl_bps_{int(tau0)}s"

if (sym0, tau0) in models and sym0 in val_datasets:
    model = models[(sym0, tau0)]
    ds_val = val_datasets[sym0]
    feat_cols = [f for f in feature_names if f in ds_val.columns]

    X_val = ds_val[feat_cols].fillna(0.0)
    y_val = ds_val[target_col]
    w_val = ds_val["w"] if "w" in ds_val.columns else pd.Series(np.ones(len(ds_val)))

    valid = ~y_val.isna()
    preds = model.predict(X_val[valid])
    y_true = y_val[valid].values
    w_true = w_val[valid].values

    # Quantile analysis
    N_BUCKETS = 10
    quantile_labels = [f"Q{i+1}" for i in range(N_BUCKETS)]
    order = np.argsort(preds)[::-1]  # best predicted first
    bucket_size = len(order) // N_BUCKETS

    bucket_pnl = []
    for b in range(N_BUCKETS):
        idx = order[b*bucket_size:(b+1)*bucket_size]
        wi = w_true[idx]
        pi = y_true[idx]
        wpnl = (wi * pi).sum() / wi.sum() if wi.sum() > 0 else 0.0
        bucket_pnl.append(wpnl)

    fig, ax = plt.subplots(figsize=(9, 4))
    colors = ["#1baf7a" if v > 0 else "#e34948" for v in bucket_pnl]
    ax.bar(quantile_labels, bucket_pnl, color=colors, alpha=0.85)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_xlabel("Predicted score quantile (Q1 = highest predicted PnL)")
    ax.set_ylabel("Realized weighted PnL (bps)")
    ax.set_title(f"Predicted vs Realized PnL by Quantile [{sym0.upper()}, τ={int(tau0)}s]")
    plt.tight_layout()
    plt.savefig("quantile_pnl.png", dpi=120, bbox_inches="tight")
    plt.show()
    print("Saved: quantile_pnl.png")
else:
    print("No validation data or model not available — skipping quantile chart")


## 10. Custom Pipeline Demo

Demonstrate the **composable** design: swap sampler, add/remove features, change transforms —
all without touching any pipeline logic code.


In [ ]:
# Example: time-clock sampler + only 2 feature blocks + no transforms
custom_config = PipelineConfig(
    features=[
        PriceZScoreFeature(windows_s=(60.0,), clip=3.0),   # just 60s z-score
        TradePressureFeature(windows_s=(30.0,)),             # just 30s pressure
    ],
    transforms=TransformLayer([]),   # no transforms
    sampler=TimeClock(interval_s=30.0),  # 30-second bars
    labeler=MarkoutLabeler(tau_s=[30.0]),
    apply_transforms=False,
)

custom_builder = DatasetBuilder(custom_config)
print(f"Custom features: {custom_builder.feature_names()}")

sym0 = SYMBOLS[0]
if sym0 in train_datasets:
    trades_tr, bbo_tr, liq_b_tr, liq_bb_tr = load_data(DATA_DIR, sym0, split="train", limit=LIMIT)
    ds_custom = custom_builder.build(trades_tr, bbo_tr, liq_b_tr, liq_bb_tr, symbol=sym0)
    print(f"Custom dataset: {len(ds_custom):,} rows × {len(custom_builder.feature_names())} features")
    display(ds_custom.head(3))


## Summary

| Component | Implementation |
|-----------|---------------|
| **Feature blocks** | Declarative list: `features = [BlockA, BlockB, ...]` |
| **Transforms** | Composable operators: `log`, `zscore`, `clamp`, `normalize` |
| **Sampler** | Swap-able: `VolumeClock` / `TimeClock` / `TradeClock` |
| **Labeler** | Markout of executed trade at τ seconds (maker PnL in bps) |
| **Dataset builder** | Single `DatasetBuilder.build()` call — deterministic & reproducible |
| **Model training** | `train_model(dataset, features, ModelSpec(...))` — fully decoupled |

### Functional pipeline graph

```
raw market data
  → PipelineState (sorted arrays for O(log n) causal lookups)
  → FeatureBlocks: TOB volume, price z-score, vol, spread, trade flow, liq pressure, time
  → TransformLayer: log → zscore → clamp (declarative, composable)
  → Sampler: volume-clock (or time-clock / trade-clock)
  → MarkoutLabeler: pnl_bps_{τ}s = -s*(mid_τ - price)/price*10000 + 0.5 bps
  → DatasetOutput: pd.DataFrame (features + targets + metadata)
  → train_model(ModelSpec) → LightGBM / Ridge / XGBoost
  → evaluate_model → IC, weighted_corr, WMSE
```
